## Antyanuprasa

In [85]:
import re
from collections import Counter

class AlankaramChecker:
    TELUGU_VOWELS = {
        'అ': 'a', 'ఆ': 'A', 'ఇ': 'i', 'ఈ': 'I', 'ఉ': 'u', 'ఊ': 'U',
        'ఋ': 'R', 'ౠ': 'RR', 'ఎ': 'e', 'ఏ': 'E', 'ఐ': 'ai',
        'ఒ': 'o', 'ఓ': 'O', 'ఔ': 'au', 'అం': 'aM', 'అః': 'aH'
    }
    TELUGU_CONSONANTS = {
        'క': 'ka', 'ఖ': 'kha', 'గ': 'ga', 'ఘ': 'gha', 'ఙ': 'Na',
        'చ': 'ca', 'ఛ': 'cha', 'జ': 'ja', 'ఝ': 'jha', 'ఞ': 'na',
        'ట': 'Ta', 'ఠ': 'Tha', 'డ': 'Da', 'ఢ': 'Dha', 'ణ': 'Na',
        'త': 'ta', 'థ': 'tha', 'ద': 'da', 'ధ': 'dha', 'న': 'na',
        'ప': 'pa', 'ఫ': 'pha', 'బ': 'ba', 'భ': 'bha', 'మ': 'ma',
        'య': 'ya', 'ర': 'ra', 'ల': 'la', 'వ': 'va',
        'శ': 'Sa', 'ష': 'sha', 'స': 'sa', 'హ': 'ha', 'ళ': 'La', 'క్ష': 'kSa', 'ఱ': 'Ra'
    }
    TELUGU_MATRAS = {
        'ా': 'A', 'ి': 'i', 'ీ': 'I', 'ు': 'u', 'ూ': 'U',
        'ృ': 'R', 'ౄ': 'RR', 'ె': 'e', 'ే': 'E', 'ై': 'ai',
        'ొ': 'o', 'ో': 'O', 'ౌ': 'au', '్': '', 'ం': 'M', 'ః': 'H'
    }

    DEVANAGARI_VOWELS = {
        'अ': 'a', 'आ': 'A', 'इ': 'i', 'ई': 'I', 'उ': 'u', 'ऊ': 'U',
        'ऋ': 'R', 'ॠ': 'RR', 'ए': 'e', 'ऐ': 'ai',
        'ओ': 'o', 'औ': 'au', 'अं': 'aM', 'अः': 'aH'
    }
    DEVANAGARI_CONSONANTS = {
        'क': 'ka', 'ख': 'kha', 'ग': 'ga', 'घ': 'gha', 'ङ': 'Na',
        'च': 'ca', 'छ': 'cha', 'ज': 'ja', 'झ': 'jha', 'ञ': 'na',
        'ट': 'Ta', 'ठ': 'Tha', 'ड': 'Da', 'ढ': 'Dha', 'ण': 'Na',
        'त': 'ta', 'थ': 'tha', 'द': 'da', 'ध': 'dha', 'न': 'na',
        'प': 'pa', 'फ': 'pha', 'ब': 'ba', 'भ': 'bha', 'म': 'ma',
        'य': 'ya', 'र': 'ra', 'ल': 'la', 'व': 'va',
        'श': 'Sa', 'ष': 'sha', 'स': 'sa', 'ह': 'ha', 'ळ': 'La', 'क्ष': 'kSa'
    }
    DEVANAGARI_MATRAS = {
        'ा': 'A', 'ि': 'i', 'ీ': 'I', 'ु': 'u', 'ू': 'U',
        'ृ': 'R', 'ॄ': 'RR', 'े': 'e', 'ै': 'ai',
        'ो': 'o', 'ौ': 'au', '्': '', 'ं': 'M', 'ः': 'H'
    }

    def detect_script(self, text):
        if re.search(r'[\u0C00-\u0C7F]', text):
            return 'telugu'
        elif re.search(r'[\u0900-\u097F]', text):
            return 'devanagari'
        else:
            return 'hk'

    def transliterate_to_hk(self, text):
        script = self.detect_script(text)
        if script == 'hk':
            return text
        
        vowels = self.TELUGU_VOWELS if script == 'telugu' else self.DEVANAGARI_VOWELS
        consonants = self.TELUGU_CONSONANTS if script == 'telugu' else self.DEVANAGARI_CONSONANTS
        matras = self.TELUGU_MATRAS if script == 'telugu' else self.DEVANAGARI_MATRAS

        hk_result = []
        i = 0
        n = len(text)
        while i < n:
            char = text[i]
            if char in vowels:
                hk_result.append(vowels[char])
                i += 1
            elif char in consonants:
                # Strip trailing 'a' from base consonant mapping to handle matras/pollu cleanly
                base = consonants[char][:-1]
                i += 1
                has_explicit_vowel_or_pollu = False
                while i < n and text[i] in matras:
                    m = matras[text[i]]
                    has_explicit_vowel_or_pollu = True
                    if m:
                        base += m
                    i += 1
                # If no matra or pollu followed, apply inherent 'a'
                if not has_explicit_vowel_or_pollu:
                    base += 'a'
                hk_result.append(base)
            else:
                hk_result.append(char)
                i += 1
        return "".join(hk_result)

    def _normalize_phonetic_sound_hk(self, akshara):
        norm = akshara
        replacements = {
            'Sa': 'sa', 'sha': 'sa',
            'La': 'la', 'Ra': 'ra',
            'Na': 'na', 'm': 'M',
        }
        for k, v in replacements.items():
            norm = norm.replace(k, v)
        return norm

    def get_aksharas(self, text):
        hk_text = self.transliterate_to_hk(text)
        # Regex captures standard syllables as well as half-consonants/pollu-ending chunks safely
        pattern = r'([bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ]+(?:[aAiIuURDeEoOMH]*))'
        matches = re.findall(pattern, hk_text)
        
        return [self._normalize_phonetic_sound_hk(a) for a in matches if a.strip()]

    def check_antyanuprasa(self, lines):
        if len(lines) < 2:
            return None
        results = []
        for i in range(len(lines) - 1):
            aksharas1 = self.get_aksharas(lines[i])
            aksharas2 = self.get_aksharas(lines[i+1])
            if not aksharas1 or not aksharas2:
                continue
            match_count = 0
            min_len = min(len(aksharas1), len(aksharas2))
            for j in range(1, min_len + 1):
                if aksharas1[-j] == aksharas2[-j]:
                    match_count += 1
                else:
                    break
            if match_count > 0:
                results.append({
                    "alankaram": "Antyanuprasa Alankaram",
                    "trigger": f"Max matching ending syllables ({match_count})",
                    "position": f"Lines {i+1} and {i+2}",
                    "match_length": match_count
                })
        return results if results else None

    def check_vrutyanuprasa(self, lines, min_repeats=3):
        results = []
        for line_idx, line in enumerate(lines):
            hk_text = self.transliterate_to_hk(line)
            aksharas = self.get_aksharas(line)
            consonant_clusters = re.findall(r'[bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ]+', hk_text)
            
            cluster_counts = Counter(consonant_clusters)
            akshara_counts = Counter(aksharas)
            
            for cluster, count in cluster_counts.items():
                if count >= min_repeats and len(cluster) > 0:
                    results.append({
                        "alankaram": "Vrutyanuprasa Alankaram",
                        "trigger": f"Consonant cluster/pollu unit '{cluster}' repeated {count} times",
                        "position": f"Line {line_idx + 1}",
                        "repeat_count": count
                    })
            for aks, count in akshara_counts.items():
                if count >= min_repeats and len(aks) > 0:
                    results.append({
                        "alankaram": "Vrutyanuprasa Alankaram",
                        "trigger": f"Syllable '{aks}' repeated {count} times",
                        "position": f"Line {line_idx + 1}",
                        "repeat_count": count
                    })
        return results if results else None

    def check_all(self, lines):
        all_results = []
        antya = self.check_antyanuprasa(lines)
        vrutya = self.check_vrutyanuprasa(lines)
        if antya: all_results.extend(antya)
        if vrutya: all_results.extend(vrutya)
        return all_results if all_results else None

In [ ]:
antyanuprasa_examples = [
        '''శ్రీరఘురామ !
        చారుతులసీ దళధామ !'''
        '''కరుణా పయోనిధీ !
        శరణాగత విరోధీ !''',
        '''ధరలోని శౌర్యము !
        కరమొప్పు ధైర్యము !''',
        '''కువలయ విలాస !
        భవరోగ వినాశ !''',
        '''సరసిజ నయనా !
        సురనుత లలనా !''',
        '''మురమథన కంసా !
        దరహాస ప్రకాశా !''',
        '''అమరవర వందితా !
        సమరవిజయ గీతా !''',
        '''ధరణీరమణ రాఘవా !
        కరుణాలవాల మేఘా !''',
        '''విమలగుణ ధామా !
        కమలాక్ష రామారామా !''',
        '''శ్రీకర భూపా !
        లోకవిలాపా !''',
        '''అందాల చందమామ
        అందమంటే పంతమా
        ఓసారి చూడమంటే ఏంటి సాకు దూరమా''',
        '''తెలుసా మనసా
        నిను చూసి తారల్లే మెరిసా
        తెలుసా వరసా
        నీతోనే ఉండాలని ఆశ
        '''
    ]

checker = AlankaramChecker()

for example in antyanuprasa_examples:
    lines = example.split('\n')
    result = checker.check_all(lines)
    if result:
        print(f"Example:\n{example}\nDetected Alankaram:\n{result}\n")
    else:
        print(f"Example:\n{example}\nNo Alankaram detected.\n")

Example:
శ్రీరఘురామ !
        చారుతులసీ దళధామ !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': 'Max matching ending syllables (1)', 'position': 'Lines 1 and 2', 'match_length': 1}]

Example:
కరుణా పయోనిధీ !
        శరణాగత విరోధీ !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': 'Max matching ending syllables (1)', 'position': 'Lines 1 and 2', 'match_length': 1}]

Example:
ధరలోని శౌర్యము !
        కరమొప్పు ధైర్యము !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': 'Max matching ending syllables (2)', 'position': 'Lines 1 and 2', 'match_length': 2}]

Example:
కువలయ విలాస !
        భవరోగ వినాశ !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': 'Max matching ending syllables (1)', 'position': 'Lines 1 and 2', 'match_length': 1}]

Example:
సరసిజ నయనా !
        సురనుత లలనా !
Detected Alankaram:
[{'alankaram': 'Antyanuprasa Alankaram', 'trigger': 'Max matching ending syllables (1)', 'position': 'Lin

In [90]:
manonethram_vrutyanuprasa_examples = [
    "ఉత్తుంగశబరిగిరిశృంగ నిత్యనిస్సంగ మంగళాంగ పంపాతరంగ పుణ్యానుషంగ మునిహృదయజలజభృంగ",
    "జిలిబిలి పలుకుల చిలిపిగ పలికిన ఓ మైనా మైనా కిలకిల నగవుల వలపులు చిలికిన ఓ మైనా మైనా",
    "వెడవెడ సిడిముడి తడబడ నడుగిడి నడుగిడదు జడిమ నడుగిడు నెడలన్",
    "చలిపిడుగుల సడి విని జడిసిన బిడియము తడబడి నిను విడదా",
    "అడుగులు తడబడ బుడతడు వడివడి నడిచెను"
]

checker = AlankaramChecker()

for i, example in enumerate(manonethram_vrutyanuprasa_examples, 1):
    lines = [example] # Passing as a single line or split if needed
    result = checker.check_all(lines)
    print(f"Example {i}:\n{example}")
    if result:
        print(f"Detected Alankaram: {result}\n")
    else:
        print("No Alankaram detected.\n")

Example 1:
ఉత్తుంగశబరిగిరిశృంగ నిత్యనిస్సంగ మంగళాంగ పంపాతరంగ పుణ్యానుషంగ మునిహృదయజలజభృంగ
Detected Alankaram: [{'alankaram': 'Vrutyanuprasa Alankaram', 'trigger': "Consonant cluster/pollu unit 'n' repeated 4 times", 'position': 'Line 1', 'repeat_count': 4}, {'alankaram': 'Vrutyanuprasa Alankaram', 'trigger': "Syllable 'ni' repeated 3 times", 'position': 'Line 1', 'repeat_count': 3}]

Example 2:
జిలిబిలి పలుకుల చిలిపిగ పలికిన ఓ మైనా మైనా కిలకిల నగవుల వలపులు చిలికిన ఓ మైనా మైనా
Detected Alankaram: [{'alankaram': 'Vrutyanuprasa Alankaram', 'trigger': "Consonant cluster/pollu unit 'l' repeated 12 times", 'position': 'Line 1', 'repeat_count': 12}, {'alankaram': 'Vrutyanuprasa Alankaram', 'trigger': "Consonant cluster/pollu unit 'p' repeated 4 times", 'position': 'Line 1', 'repeat_count': 4}, {'alankaram': 'Vrutyanuprasa Alankaram', 'trigger': "Consonant cluster/pollu unit 'k' repeated 5 times", 'position': 'Line 1', 'repeat_count': 5}, {'alankaram': 'Vrutyanuprasa Alankaram', 'trigger': "Con